# Import libraries and environment keys

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

# Langchain Implementations

## OpenAI Models

In [7]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

# Model parameters
temperature = 0.6
max_Tokens = 1000
model_name = "gpt-4o"

# Initialize the model
chat = ChatOpenAI(proxy_model_name=model_name, max_tokens=max_Tokens, temperature=temperature)

prompt = """What is the capital of France?"""
response = chat.invoke(prompt)
print(response)


content='The capital of France is **Paris**.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 4, 'engine_ttft_ms': 29, 'engine_ttlt_ms': 67, 'pre_inference_ms': 73, 'service_tbt_ms': 4, 'service_ttft_ms': 156, 'service_ttlt_ms': 185, 'total_duration_ms': 118, 'user_visible_ttft_ms': 83}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_49e2bef596', 'id': 'chatcmpl-DsSOkrqxkgHZMqbe8t4LITUvKYa8Y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019edfcd-7a76-7b20-9d02-f27723c6fe44-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_toke

In [8]:
print(response.content)

The capital of France is **Paris**.


## OpenAI Reasoning Models
OpenAI Reasoning models, such as GPT-5, do not accept parameters such as `temperature` or `max_Tokens`. Instead, they use `verbosity` as a proxy for `temperature`, with values `low`, `medium` and `high`, and `reasoning_effort`, with values `minimal`, `low`, `medium` and `high`.

In [9]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
import time

# Model parameters
model_name = "gpt-5.4"

# Iterating on verbosity
verbosity_list = ["low", "medium", "high"]
reasoning_effort_list = ["minimal", "low", "medium", "high"]

#Fixed verbosity
for reasoning_effort in reasoning_effort_list:
    start_time = time.time()
    chat = ChatOpenAI(proxy_model_name=model_name, verbosity="medium", reasoning_effort=reasoning_effort)
    prompt = f"""What is Green Theorem?"""
    response = chat.invoke(prompt)
    elapsed_time = time.time() - start_time
    text_response = response.content[:200]  # Get the first 200 characters of the response
    print(f"Reasoning Effort: {reasoning_effort}\nResponse: {text_response} (pruned) \nTime: {elapsed_time:.2f}s\n")
    print('---'*10)

# Fixed reasoning effort
for verbosity in verbosity_list:
    start_time = time.time()
    chat = ChatOpenAI(proxy_model_name=model_name, verbosity=verbosity, reasoning_effort="medium")
    prompt = f"""What is Green Theorem?"""
    response = chat.invoke(prompt)
    elapsed_time = time.time() - start_time
    text_response = response.content
    len_response = len(text_response)
    print(f"Verbosity: {verbosity}\nResponse: {text_response[:200]} (pruned) \nTime: {elapsed_time:.2f}s \nTotal length: {len_response}\n")
    print('---'*10)


Reasoning Effort: minimal
Response: Green’s Theorem is a result in vector calculus that relates a line integral around a closed curve to a double integral over the region inside the curve.

## Statement

If \(C\) is a positively oriente (pruned) 
Time: 9.44s

------------------------------
Reasoning Effort: low
Response: **Green’s Theorem** is a result in vector calculus that relates a **line integral around a closed curve** to a **double integral over the region inside the curve**.

## Statement

If \(C\) is a positi (pruned) 
Time: 7.08s

------------------------------
Reasoning Effort: medium
Response: **Green’s Theorem** is a result in vector calculus that connects:

- a **line integral** around a closed curve, and
- a **double integral** over the region enclosed by that curve.

## Statement

If \( (pruned) 
Time: 8.07s

------------------------------
Reasoning Effort: high
Response: **Green’s Theorem** is a result from vector calculus that connects a **line integral around a close

## OpenAI Responses API through LangChain
SAP's LangChain `ChatOpenAI` wrapper can route GPT-5.* model calls through OpenAI's Responses API by setting `use_responses_api=True`. In the current SAP AI SDK wrapper, clear the default `n=1` value before invoking because the native Responses API client does not accept an `n` keyword argument. The call still returns a LangChain `AIMessage`; use `response.text` for the final text, while the typed Responses API content blocks remain available through `response.content`. Metadata and token usage are exposed through `response.response_metadata` and `response.usage_metadata`.

In [14]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

# Model parameters
model_name = "gpt-5.5"

# use_responses_api=True forces the LangChain wrapper to use the Responses API.
# reasoning_effort and verbosity are converted to Responses API payload fields.
responses_chat = ChatOpenAI(
    proxy_model_name=model_name,
    use_responses_api=True,
    reasoning_effort="low",
    verbosity="medium",
)

# SAP's wrapper defaults n to 1 for Chat Completions compatibility.
# The native Responses API client rejects `n`, so clear it before invoke().
responses_chat.n = None

prompt = "In one sentence, explain what SAP AI Core provides inside SAP BTP."
responses_message = responses_chat.invoke(prompt)

print(responses_message.text)
print(f"\nResponse ID: {responses_message.response_metadata.get('id', 'not returned')}")
print(f"Usage metadata: {responses_message.usage_metadata}")


SAP AI Core is a service in SAP BTP that lets you build, deploy, run, and manage AI/ML workloads and generative AI scenarios in a scalable, enterprise-ready way.

Response ID: resp_0044c9d47937f6a1006a35355c8a08819791d8383e96817d69
Usage metadata: {'input_tokens': 21, 'output_tokens': 42, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


The same wrapper also supports `use_previous_response_id=True` or a `previous_response_id` invocation argument for stateful follow-up turns when the selected deployment supports Responses API conversation state.

## Amazon and Anthropic Models

In [9]:
from gen_ai_hub.proxy.langchain.amazon import ChatBedrock

# Model parameters
temperature = 0.6
max_tokens = 1000  # Note: variable name should match parameter name
model_name = "anthropic--claude-4.5-sonnet"

# Initialize the model
chat = ChatBedrock(
    model_name=model_name,  # Use model_name parameter instead of model_id
    temperature=temperature,
    model_kwargs={
        "max_tokens": max_tokens,  # For Anthropic models
    }
)

prompt = """What is the capital of France?"""
response = chat.invoke(prompt)
print(response)


/var/folders/wc/q7_36ld508x9bwjncqlgk8dw0000gn/T/ipykernel_71112/1632046624.py:9: UserWarning: WARNING! client_params is not default parameter.
                client_params was transferred to model_kwargs.
                Please confirm that client_params is what you intended.
  chat = ChatBedrock(


content='The capital of France is Paris.' additional_kwargs={'usage': {'prompt_tokens': 14, 'completion_tokens': 10, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 24}, 'stop_reason': 'end_turn', 'model_id': 'anthropic.claude-sonnet-4-5-20250929-v1:0'} response_metadata={'usage': {'prompt_tokens': 14, 'completion_tokens': 10, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 24}, 'stop_reason': 'end_turn', 'model_id': 'anthropic.claude-sonnet-4-5-20250929-v1:0', 'model_provider': 'bedrock', 'model_name': 'anthropic.claude-sonnet-4-5-20250929-v1:0'} id='lc_run--019c2ce2-06a4-73c2-848c-38caea90dfdf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}


In [10]:
print(response.content)

The capital of France is Paris.


## Google Vertex AI Models

In [ ]:
from gen_ai_hub.proxy.langchain.google_genai import ChatGoogleGenerativeAI
# from gen_ai_hub.proxy.native.google_genai import GenerativeModel

# Model parameters
temperature = 0.6
max_tokens = 1000
model_name = "gemini-2.5-pro"

# Initialize the model
chat = ChatGoogleGenerativeAI(
    model=model_name,  # Use model_name parameter instead of model_id
    temperature=temperature,
    max_output_tokens=max_tokens  # For Vertex AI models
)

prompt = """What is the capital of France?"""
response = chat.invoke(prompt)
print(response)

content='The capital of France is **Paris**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c2cdf-efc9-7d22-8792-a1e3d8c0a715-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 250, 'total_tokens': 257, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 242}}


In [4]:
print(response.content)

The capital of France is **Paris**.
